# Policy Analysis Figures

Workflow: edit the function → run the show cell to preview → run the save cell.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
def find_project_root():
    path = Path.cwd()
    for _ in range(5):
        if (path / "code").is_dir() and (path / "results").is_dir():
            return path
        path = path.parent
    return Path.cwd()


PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results" / "policy_analysis"
FIGURES_DIR = RESULTS_DIR / "figs"
FIGURES_DIR.mkdir(exist_ok=True)

print(f"Project : {PROJECT_ROOT}")
print(f"Output  : {RESULTS_DIR}")

In [ ]:
# -- column names (hard-coded) --------------------------------------------

PERCENT_METRICS = [
    "sale_price",
    "rent",
    "avg_household_net_worth",
]

POINT_METRICS = [
    "price_volatility",
    "household_net_worth_gini",
]

COUNT_SHARES = [
    "owner_occupier_marginal_count_share",
    "private_landlord_marginal_count_share",
    "institution_marginal_count_share",
]

VALUE_SHARES = [
    "owner_occupier_marginal_value_share",
    "private_landlord_marginal_value_share",
    "institution_marginal_value_share",
]

STOCK_SHARES = [
    "owner_occupier_stock_share",
    "private_landlord_stock_share",
    "institution_stock_share",
]

In [ ]:
# -- labels (edit these to change titles / axis labels) --------------------

POLICY_LABELS = {
    "rate-up": "Rate Up",
    "rate-down": "Rate Down",
    "ltv-tighten": "LTV Tighten",
    "ltv-loosen": "LTV Loosen",
    "recession-credit-crunch": "Recession + Credit Crunch",
    "recession-easing-crunch": "Recession + Easing Crunch",
    "boom-credit-expansion": "Boom + Credit Expansion",
}

METRIC_LABELS = {
    "sale_price": "Sale Price",
    "rent": "Rent",
    "avg_household_net_worth": "Avg Household Net Worth",
    "price_volatility": "Price Volatility",
    "household_net_worth_gini": "Wealth Gini",
    "owner_occupier_marginal_count_share": "Owner-Occupier (count)",
    "private_landlord_marginal_count_share": "Landlord (count)",
    "institution_marginal_count_share": "Institution (count)",
    "owner_occupier_marginal_value_share": "Owner-Occupier (value)",
    "private_landlord_marginal_value_share": "Landlord (value)",
    "institution_marginal_value_share": "Institution (value)",
    "owner_occupier_stock_share": "Owner-Occupier Stock",
    "private_landlord_stock_share": "Landlord Stock",
    "institution_stock_share": "Institution Stock",
}

In [ ]:
# -- load data -------------------------------------------------------------

csv_path = RESULTS_DIR / "responses.csv"
if not csv_path.exists():
    raise FileNotFoundError(f"{csv_path} not found. Run code/policy_analysis.py first.")

responses = pd.read_csv(csv_path)
POLICIES = sorted([p for p in responses["policy"].unique() if p != "policy"])

print(f"Policies ({len(POLICIES)}): {POLICIES}")
print(f"Rows: {len(responses)}")

In [ ]:
# -- summarise -------------------------------------------------------------


def summarise(responses):
    """Pointwise means and approximate 95% confidence intervals."""
    metrics = PERCENT_METRICS + POINT_METRICS + COUNT_SHARES + VALUE_SHARES + STOCK_SHARES
    long_data = responses.melt(
        id_vars=["policy", "seed", "event_month"],
        value_vars=metrics,
        var_name="metric",
        value_name="response",
    )
    summary = (
        long_data.groupby(["policy", "event_month", "metric"])["response"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    half_width = 1.96 * summary["std"] / np.sqrt(summary["count"])
    summary["lower"] = summary["mean"] - half_width
    summary["upper"] = summary["mean"] + half_width
    return summary


summary = summarise(responses)
print(f"Summary rows: {len(summary)}")

---
## Figure 1: Per-Policy Response

Single function driving all per-policy figures.
Edit the layout/metrics here, then run show + save for each policy.


In [ ]:
# 1a. Function


def plot_policy_response(policy_name, summary, figsize=(10, 9), sharex=True):
    """3x1 grid: sale price, rent, marginal-pricer count shares."""
    policy_data = summary[summary["policy"] == policy_name]
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=sharex)

    def plot_band(ax, metric, label=None, alpha=0.2):
        values = policy_data[policy_data["metric"] == metric].sort_values("event_month")
        line = ax.plot(values["event_month"], values["mean"], label=label)[0]
        ax.fill_between(
            values["event_month"],
            values["lower"],
            values["upper"],
            color=line.get_color(),
            alpha=alpha,
        )

    plot_band(axes[0], "sale_price")
    axes[0].set_ylabel("Sale price response (%)")

    plot_band(axes[1], "rent")
    axes[1].set_ylabel("Rent response (%)")

    plot_band(axes[2], "owner_occupier_marginal_count_share", "Owner-occupier", 0.12)
    plot_band(axes[2], "private_landlord_marginal_count_share", "Landlord", 0.12)
    plot_band(axes[2], "institution_marginal_count_share", "Institution", 0.12)
    axes[2].set_ylabel("Marginal-pricer proxy (pp)")
    axes[2].set_xlabel("Months relative to permanent policy shift")
    axes[2].legend(fontsize=8)

    for ax in axes:
        ax.axhline(0, color="black", linewidth=0.8)
        ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_xlim(-60, 120)
        ax.grid(alpha=0.25)

    label = POLICY_LABELS.get(policy_name, policy_name.replace("-", " ").title())
    fig.suptitle(label, fontweight="bold", fontsize=13)
    fig.tight_layout()
    return fig

### A. Rate Up


In [ ]:
# Ai. Show
fig_rate_up = plot_policy_response("rate-up", summary)
plt.show()

In [ ]:
# Aii. Save
fig_rate_up.savefig(FIGURES_DIR / "rate_up_response.png", dpi=300, bbox_inches="tight")
plt.close(fig_rate_up)
print(f"Saved: rate_up_response.png")

### B. Rate Down


In [ ]:
# Bi. Show
fig_rate_down = plot_policy_response("rate-down", summary)
plt.show()

In [ ]:
# Bii. Save
fig_rate_down.savefig(FIGURES_DIR / "rate_down_response.png", dpi=300, bbox_inches="tight")
plt.close(fig_rate_down)
print(f"Saved: rate_down_response.png")

### C. LTV Tighten


In [ ]:
# Ci. Show
fig_ltv_tighten = plot_policy_response("ltv-tighten", summary)
plt.show()

In [ ]:
# Cii. Save
fig_ltv_tighten.savefig(
    FIGURES_DIR / "ltv_tighten_response.png", dpi=300, bbox_inches="tight"
)
plt.close(fig_ltv_tighten)
print(f"Saved: ltv_tighten_response.png")

### D. LTV Loosen


In [ ]:
# Di. Show
fig_ltv_loosen = plot_policy_response("ltv-loosen", summary)
plt.show()

In [ ]:
# Dii. Save
fig_ltv_loosen.savefig(
    FIGURES_DIR / "ltv_loosen_response.png", dpi=300, bbox_inches="tight"
)
plt.close(fig_ltv_loosen)
print(f"Saved: ltv_loosen_response.png")

### E. Recession + Credit Crunch


In [ ]:
# Ei. Show
fig_recession_credit_crunch = plot_policy_response("recession-credit-crunch", summary)
plt.show()

In [ ]:
# Eii. Save
fig_recession_credit_crunch.savefig(
    FIGURES_DIR / "recession_credit_crunch_response.png", dpi=300, bbox_inches="tight"
)
plt.close(fig_recession_credit_crunch)
print(f"Saved: recession_credit_crunch_response.png")

### F. Recession + Easing Crunch


In [ ]:
# Fi. Show
fig_recession_easing_crunch = plot_policy_response("recession-easing-crunch", summary)
plt.show()

In [ ]:
# Fii. Save
fig_recession_easing_crunch.savefig(
    FIGURES_DIR / "recession_easing_crunch_response.png", dpi=300, bbox_inches="tight"
)
plt.close(fig_recession_easing_crunch)
print(f"Saved: recession_easing_crunch_response.png")

### G. Boom + Credit Expansion


In [ ]:
# Gi. Show
fig_boom_credit_expansion = plot_policy_response("boom-credit-expansion", summary)
plt.show()

In [ ]:
# Gii. Save
fig_boom_credit_expansion.savefig(
    FIGURES_DIR / "boom_credit_expansion_response.png", dpi=300, bbox_inches="tight"
)
plt.close(fig_boom_credit_expansion)
print("Saved: boom_credit_expansion_response.png")

---
## Figure 2: Marginal-Pricer Comparison


In [ ]:
# 2a. Function


def plot_marginal_comparison(policy_names, summary, figsize=(9, 3 * 6 + 1)):
    """Nx1 grid comparing marginal-pricer proxy responses across policies."""
    n = len(policy_names)
    fig, axes = plt.subplots(n, 1, figsize=figsize, sharex=True)
    if n == 1:
        axes = [axes]

    metrics = {
        "owner_occupier_marginal_count_share": "Owner-occupier",
        "private_landlord_marginal_count_share": "Landlord",
        "institution_marginal_count_share": "Institution",
    }

    for ax, policy in zip(axes, policy_names):
        policy_data = summary[summary["policy"] == policy]
        for metric, label in metrics.items():
            values = policy_data[policy_data["metric"] == metric].sort_values("event_month")
            line = ax.plot(values["event_month"], values["mean"], label=label)[0]
            ax.fill_between(
                values["event_month"].to_numpy(),
                values["lower"].to_numpy(),
                values["upper"].to_numpy(),
                color=line.get_color(),
                alpha=0.12,
            )

        ax.axhline(0, color="black", linewidth=0.8)
        ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_xlim(-60, 120)
        ax.set_ylabel("Marginal-pricer proxy (pp)")
        ax.set_title(POLICY_LABELS.get(policy, policy.replace("-", " ").title()))
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)

    axes[-1].set_xlabel("Months relative to permanent policy shift")
    fig.tight_layout()
    return fig

In [ ]:
# 2b. Show
fig_mp = plot_marginal_comparison(POLICIES, summary)
plt.show()

In [ ]:
# 2c. Save
fig_mp.savefig(
    FIGURES_DIR / "marginal_pricer_comparison.png", dpi=300, bbox_inches="tight"
)
plt.close(fig_mp)
print("Saved: marginal_pricer_comparison.png")